# Smart Workforce & Resource Scheduling Optimization
## Notebook 1: Data Loading and Cleaning

In this notebook I load the call center workforce dataset, check its quality and save a clean version for the analysis.

Data source: [Kaggle – Call Center Workforce Dataset (WFO, AHT, KPI)](https://www.kaggle.com/datasets/komalkomal7/call-center-workforce-dataset-wfo-aht-kpi)

The data is simulated (made with Mockaroo and adjusted with business rules). One row is one queue, at one site, in one hour.

## 1. Setup
Import the libraries and set the project paths.

In [1]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File paths
from pathlib import Path

# Display settings: show all columns when printing a DataFrame
pd.set_option("display.max_columns", None)

In [2]:
# Project paths (the notebook lives in /notebooks, so we go one level up)
PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

# Create the folders if they do not exist yet
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle dataset identifier and file name
KAGGLE_DATASET = "komalkomal7/call-center-workforce-dataset-wfo-aht-kpi"
RAW_FILE = RAW_DIR / "call_center_operations_wfo_dataset.csv"

## 2. Download the data via Kaggle API
The raw file is only downloaded if it is not already in `data/raw/`, so the notebook does not download it again every time I run it.

In [3]:
from kaggle.api.kaggle_api_extended import KaggleApi

if RAW_FILE.exists():
    print(f"File already exists: {RAW_FILE}")
else:
    api = KaggleApi()
    api.authenticate()  # reads the token from ~/.kaggle/access_token
    api.dataset_download_files(KAGGLE_DATASET, path=RAW_DIR, unzip=True)
    print(f"Downloaded to: {RAW_DIR}")

## 3. Load the raw data
I load the file into `df_raw`. I never change this table.

In [4]:
df_raw = pd.read_csv(RAW_FILE)

print(f"Rows: {df_raw.shape[0]:,} | Columns: {df_raw.shape[1]}")
df_raw.head()

Rows: 10,000 | Columns: 30


,Date,Weekday,Week_Number,Month,Year,Hour,Country,Region,Site,Queue_Name,Channel,All_Units_Filter,Queue_Units_Filter,ASU_Tier_Filter,Business_Unit,planned_Volume,offered_Volume,handled_Volume,abandoned_Volume,backlog_Volume,aHT_Seconds,handle_Time_Total,Wait_Time_Avg,shrinkage_Percent,required_FTE,agent_Count,utilization_percent,Active_Agents,Service_Level_Percent,Occupancy_Percent
0,29-09-2023,Friday,39,September,2023,3,Bulgaria,EMEA,Bulgaria Hub,INBOUND_VOICE_TIER1,Chat,Consumer,Inbound,Tier 1,Banking,464,492,484,8,8,355,171820,5.772358,28,5,6,71,4,86,89
1,27-10-2022,Thursday,43,October,2022,16,Malaysia,APAC,Kuala Lumpur,TECH_SUPPORT_EMAIL_T3,Chat,Consumer,Tech Support,Tier 2,Banking,100,101,88,13,13,613,53944,78.900990,31,1,1,75,1,82,60
2,16-10-2022,Sunday,41,October,2022,15,Kenya,EMEA,Kenya Hub,EMEA_CHAT_SUPPORT_T1,Email,Consumer,Support,Tier 1,Retail,91,79,66,13,13,272,17952,44.759494,18,0,0,90,0,73,71
3,23-06-2021,Wednesday,25,June,2021,12,New Zealand,APAC,New Zealand Hub,APAC_EMAIL_SUPPORT_T3,Voice,Consumer,Support,Tier 2,Banking,220,219,216,3,3,459,99144,6.287671,32,3,4,91,4,82,73
4,24-10-2021,Sunday,42,October,2021,18,Bangladesh,APAC,Bangladesh Hub,APAC_CORE_EMAIL_TIER2,Chat,Enterprise,Support,Tier 1,Telecom,484,468,444,24,24,867,384948,44.461538,27,13,17,71,12,77,73


## 4. First look at the data
I check the column types, missing values, duplicates, value ranges and the date range. The results tell me which cleaning steps I need.

In [5]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Date                   10000 non-null  str    
 1   Weekday                10000 non-null  str    
 2   Week_Number            10000 non-null  int64  
 3   Month                  10000 non-null  str    
 4   Year                   10000 non-null  int64  
 5   Hour                   10000 non-null  int64  
 6   Country                10000 non-null  str    
 7   Region                 10000 non-null  str    
 8   Site                   10000 non-null  str    
 9   Queue_Name             10000 non-null  str    
 10  Channel                10000 non-null  str    
 11  All_Units_Filter       10000 non-null  str    
 12  Queue_Units_Filter     10000 non-null  str    
 13  ASU_Tier_Filter        10000 non-null  str    
 14  Business_Unit          10000 non-null  str    
 15  planned_Volume

In [6]:
# Missing values per column (only columns with at least one missing value)
missing = df_raw.isna().sum()
print("Columns with missing values:")
print(missing[missing > 0] if missing.sum() > 0 else "None")

# Fully duplicated rows
print(f"\nDuplicate rows: {df_raw.duplicated().sum()}")

Columns with missing values:
None

Duplicate rows: 0


In [7]:
df_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
Week_Number,10000.0,25.789000,15.242425,1.0,12.00000,25.000000,39.000000,53.000000
Year,10000.0,2023.106500,1.512411,2021.0,2022.00000,2023.000000,2024.000000,2026.000000
Hour,10000.0,11.422400,6.956983,0.0,5.00000,11.000000,17.000000,23.000000
planned_Volume,10000.0,275.585700,131.131163,50.0,161.00000,276.000000,390.000000,500.000000
offered_Volume,10000.0,280.581300,132.009230,31.0,166.00000,281.000000,395.000000,530.000000
handled_Volume,10000.0,265.667600,132.229725,3.0,151.00000,266.000000,379.000000,521.000000
abandoned_Volume,10000.0,14.913700,8.974024,0.0,7.00000,15.000000,23.000000,30.000000
backlog_Volume,10000.0,14.913700,8.974024,0.0,7.00000,15.000000,23.000000,30.000000
aHT_Seconds,10000.0,539.724000,207.413418,180.0,361.00000,542.000000,719.000000,900.000000
handle_Time_Total,10000.0,143432.488800,94120.655794,801.0,67568.25000,122689.000000,203651.000000,455875.000000


In [10]:
# Number of unique values in each text (categorical) column
categorical_cols = df_raw.select_dtypes(include="str").columns
df_raw[categorical_cols].nunique().sort_values()

Channel                  3
Region                   3
ASU_Tier_Filter          3
All_Units_Filter         3
Business_Unit            5
Queue_Units_Filter       6
Weekday                  7
Month                   12
Queue_Name              20
Country                 53
Site                    58
Date                  1885
dtype: int64

In [9]:
# Data cutoff: last complete day before the data was downloaded (9 Oct 2026).
# Rows dated after this day are treated as invalid (future dates).
DATA_CUTOFF = pd.Timestamp("2026-10-08")

# Convert the text dates (format: DD-MM-YYYY) to real dates, for checking only
dates = pd.to_datetime(df_raw["Date"], format="%d-%m-%Y", errors="coerce")

print(f"Date range: {dates.min().date()} to {dates.max().date()}")
print(f"Invalid dates: {dates.isna().sum()}")
print(f"Rows after cutoff ({DATA_CUTOFF.date()}): {(dates > DATA_CUTOFF).sum()}")

Date range: 2021-01-01 to 2026-12-03
Invalid dates: 0
Rows after cutoff (2026-10-08): 30


### Findings
- No missing values, no duplicate rows and no invalid dates.
- `Date` is stored as text (DD-MM-YYYY), so I convert it to datetime.
- The column names have mixed upper and lower case. I change them all to lowercase.
- The dates run from 2021-01-01 to 2026-12-03. 30 rows are after my cutoff date (2026-10-08, the last complete day before I downloaded the data).
- `abandoned_Volume` and `backlog_Volume` have identical statistics. I confirm this row by row in section 5.
- `Wait_Time_Avg` is skewed to the right (mean 40 s, median 25 s, max 676 s), so I report medians next to the means.
- The data is a sparse sample: 10,000 rows over 1,885 dates, 58 sites and 20 queues. Most combinations of site, queue and hour appear only a few times.

## 5. Consistency checks
Before cleaning, I check three things in code:
1. Are `backlog_Volume` and `abandoned_Volume` the same column?
2. Are the rows with zero active agents also the rows with zero required FTE?
3. Do the volume and staffing columns follow basic business rules?

In [11]:
# Check 1: is backlog_Volume an exact copy of abandoned_Volume?
same_values = (df_raw["backlog_Volume"] == df_raw["abandoned_Volume"]).all()
print(f"backlog_Volume equals abandoned_Volume in every row: {same_values}")

backlog_Volume equals abandoned_Volume in every row: True


In [12]:
# Check 2: rows without staff
zero_active = df_raw["Active_Agents"] == 0
zero_required = df_raw["required_FTE"] == 0
zero_count = df_raw["agent_Count"] == 0

print(f"Rows with Active_Agents = 0: {zero_active.sum()}")
print(f"Rows with required_FTE = 0: {zero_required.sum()}")
print(f"Rows with agent_Count = 0:   {zero_count.sum()}")
print(f"Rows where all three are 0:  {(zero_active & zero_required & zero_count).sum()}")

# Do these rows still have handled contacts?
print(f"Zero-staff rows with handled_Volume > 0: {(zero_active & (df_raw['handled_Volume'] > 0)).sum()}")

Rows with Active_Agents = 0: 633
Rows with required_FTE = 0: 633
Rows with agent_Count = 0:   633
Rows where all three are 0:  633
Zero-staff rows with handled_Volume > 0: 633


In [13]:
# Check 3: basic business rules (each should hold in all 10,000 rows)
rules = {
    "offered - handled = abandoned": (df_raw["offered_Volume"] - df_raw["handled_Volume"]) == df_raw["abandoned_Volume"],
    "handled <= offered": df_raw["handled_Volume"] <= df_raw["offered_Volume"],
    "active agents <= agent count": df_raw["Active_Agents"] <= df_raw["agent_Count"],
    "handle time total = handled x AHT": df_raw["handle_Time_Total"] == df_raw["handled_Volume"] * df_raw["aHT_Seconds"],
}

for rule_name, result in rules.items():
    print(f"{rule_name:<36} {result.sum():>6,} / {len(df_raw):,} rows pass")

offered - handled = abandoned        10,000 / 10,000 rows pass
handled <= offered                   10,000 / 10,000 rows pass
active agents <= agent count         10,000 / 10,000 rows pass
handle time total = handled x AHT    10,000 / 10,000 rows pass


### Findings
- `backlog_Volume` is an exact copy of `abandoned_Volume`, so I drop it.
- 633 rows have `Active_Agents`, `required_FTE` and `agent_Count` all equal to 0. They are the same rows, and all of them still show handled contacts. I keep them, mark them with `zero_staff_flag` and leave them out of the main staffing analysis. (One of the 633 is after the cutoff, so 632 remain.)
- A real call center cannot handle contacts with zero agents, so I treat this as a question about the data, not as a real hour without staff. In Notebook 2 I test whether these are rows where the workload is below one 8-hour FTE (`required_FTE = floor(handle_Time_Total / 28,800)`).
- All four business rules hold in every row, so the volume columns agree with each other. But consistent is not the same as realistic. I check that in Notebook 2.

## 6. Cleaning
Based on the checks above, I do these steps:
1. Change the column names to lowercase.
2. Convert `date` from text to datetime.
3. Remove rows dated after the cutoff (2026-10-08).
4. Drop `backlog_volume`, because it is a copy of `abandoned_volume`.
5. Add `zero_staff_flag` for rows where `active_agents` = 0 (the same rows as `required_fte` = 0 and `agent_count` = 0). These rows still show handled contacts, so I flag them and leave them out of the main analysis. I do not delete them.

I apply everything to a copy (`df`). The raw table (`df_raw`) stays unchanged.

In [14]:
# Work on a copy, so df_raw stays unchanged
df = df_raw.copy()

# 1. Standardize column names to lowercase
df.columns = df.columns.str.lower()
print(df.columns.tolist())

['date', 'weekday', 'week_number', 'month', 'year', 'hour', 'country', 'region', 'site', 'queue_name', 'channel', 'all_units_filter', 'queue_units_filter', 'asu_tier_filter', 'business_unit', 'planned_volume', 'offered_volume', 'handled_volume', 'abandoned_volume', 'backlog_volume', 'aht_seconds', 'handle_time_total', 'wait_time_avg', 'shrinkage_percent', 'required_fte', 'agent_count', 'utilization_percent', 'active_agents', 'service_level_percent', 'occupancy_percent']


In [15]:
# 2. Convert date from text (DD-MM-YYYY) to datetime
df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y")

# 3. Remove rows dated after the cutoff
rows_before = len(df)
df = df[df["date"] <= DATA_CUTOFF].copy()

print(f"Rows removed (after {DATA_CUTOFF.date()}): {rows_before - len(df)}")
print(f"Date range now: {df['date'].min().date()} to {df['date'].max().date()}")

Rows removed (after 2026-10-08): 30
Date range now: 2021-01-01 to 2026-10-03


In [16]:
# 4. Drop the duplicated column
df = df.drop(columns="backlog_volume")

# 5. Flag rows where active_agents, required_fte and agent_count are all 0 (same rows, verified in section 5).
# The name is kept for simplicity; see the findings in section 5 for how these rows are interpreted.
df["zero_staff_flag"] = df["active_agents"] == 0

print(f"Shape after cleaning: {df.shape}")
print(f"Rows flagged as zero staff: {df['zero_staff_flag'].sum()}")

Shape after cleaning: (9970, 30)
Rows flagged as zero staff: 632


## 7. Save the clean data
The clean file is the input for the analysis notebook, the SQL queries and the Tableau dashboard.

In [17]:
CLEAN_FILE = CLEAN_DIR / "call_center_clean.csv"
df.to_csv(CLEAN_FILE, index=False)

print(f"Saved: {CLEAN_FILE}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")

Saved: ..\data\clean\call_center_clean.csv
Rows: 9,970 | Columns: 30


### Cleaning summary
| Step | Result |
|---|---|
| Raw data | 10,000 rows, 30 columns |
| Rows after cutoff removed | −30 rows |
| `backlog_volume` dropped | −1 column |
| `zero_staff_flag` added | +1 column (632 rows flagged) |
| **Clean data** | **9,970 rows, 30 columns** |

Saved to `data/clean/call_center_clean.csv`.